# Tendencias de Consumo
**Proyecto:** Reserva Inteligente de Restaurantes — Etapa 3  
**Análisis:** Productos más vendidos, ingresos por categoría y variación MoM  
**Fuente:** Data Warehouse Hive (`restaurant_dw`)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("tendencias_consumo_notebook")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "/opt/hive/data/warehouse")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionURL", 
            "jdbc:postgresql://hive-metastore-db:5432/metastore")
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName", 
            "org.postgresql.Driver")
    .config("spark.hadoop.javax.jdo.option.ConnectionUserName", "hive")
    .config("spark.hadoop.javax.jdo.option.ConnectionPassword", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("USE restaurant_dw")
print("Spark version:", spark.version)

In [ ]:
# fact_pedido tiene columnas de partición anio/mes que chocan con dim_tiempo.
# También precio_unitario choca con dim_plato. Hacemos select explícito.
fact_pedido = spark.table("fact_pedido").select(
    "id_tiempo", "id_restaurante", "id_usuario", "id_plato",
    "id_tipo_pedido", "id_estado_pedido",
    "id_pedido_origen", "id_plato_origen",
    "cantidad",
    F.col("precio_unitario").alias("precio_unitario_venta"),
    "subtotal", "precio_total_pedido",
)

# Renombramos id y nombre en cada dimensión para evitar ambigüedad en los joins
dim_tiempo       = spark.table("dim_tiempo").withColumnRenamed("id", "id_tiempo")
dim_plato        = spark.table("dim_plato").withColumnRenamed("id", "id_plato").withColumnRenamed("nombre", "nombre_plato")
dim_restaurante  = spark.table("dim_restaurante").withColumnRenamed("id", "id_restaurante").withColumnRenamed("nombre", "nombre_restaurante")
dim_estado       = spark.table("dim_estado_pedido").withColumnRenamed("id", "id_estado_pedido").withColumnRenamed("nombre", "estado_nombre")

print(f"fact_pedido: {fact_pedido.count()} filas")
fact_pedido.printSchema()

## 1. Ingresos por mes y categoría

In [ ]:
df_mes_cat = (
    fact_pedido
    .join(dim_tiempo,       "id_tiempo")
    .join(dim_plato,         "id_plato")
    .join(dim_restaurante,   "id_restaurante")
    .join(dim_estado,        "id_estado_pedido")
    .filter(F.col("estado_nombre") == "completado")
    .groupBy(
        F.col("anio"), F.col("mes"), F.col("nombre_mes"),
        F.col("categoria"),
        F.col("nombre_restaurante").alias("restaurante"),
    )
    .agg(
        F.countDistinct("id_pedido_origen").alias("total_pedidos"),
        F.sum("cantidad").alias("unidades_vendidas"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
        F.round(F.avg("precio_unitario_venta"), 2).alias("precio_promedio"),
    )
    .orderBy("anio", "mes", F.desc("ingresos"))
)

df_mes_cat.show(20, truncate=False)

## 2. Variación MoM (mes a mes) por categoría

In [ ]:
w = Window.partitionBy("categoria", "restaurante").orderBy("anio", "mes")

df_mom = (
    df_mes_cat
    .withColumn("ingresos_mes_anterior", F.lag("ingresos", 1).over(w))
    .withColumn("variacion_mom_pct",
        F.when(F.col("ingresos_mes_anterior") > 0,
            F.round(
                (F.col("ingresos") - F.col("ingresos_mes_anterior"))
                * 100.0 / F.col("ingresos_mes_anterior"), 2
            )
        ).otherwise(F.lit(None))
    )
    .drop("ingresos_mes_anterior")
)

df_mom.select(
    "anio", "mes", "nombre_mes", "categoria",
    "restaurante", "ingresos", "variacion_mom_pct"
).show(20, truncate=False)

## 3. Top 5 platos más vendidos por mes

In [ ]:
df_top_base = (
    fact_pedido
    .join(dim_tiempo,  "id_tiempo")
    .join(dim_plato,   "id_plato")
    .join(dim_estado,  "id_estado_pedido")
    .filter(F.col("estado_nombre") == "completado")
    .groupBy(
        F.col("anio"), F.col("mes"), F.col("nombre_mes"),
        F.col("nombre_plato").alias("plato"),
        F.col("categoria"),
    )
    .agg(
        F.sum("cantidad").alias("unidades_vendidas"),
        F.round(F.sum("subtotal"), 2).alias("ingresos"),
    )
)

w_rank = Window.partitionBy("anio", "mes").orderBy(F.desc("unidades_vendidas"))

df_top = (
    df_top_base
    .withColumn("rank_mes", F.rank().over(w_rank))
    .filter(F.col("rank_mes") <= 5)
    .orderBy("anio", "mes", "rank_mes")
)

df_top.show(30, truncate=False)

## 4. Guardar resultados en Hive

In [ ]:
import shutil, os

WAREHOUSE = "/opt/hive/data/warehouse/restaurant_dw.db"

def save_table(df, nombre):
    tabla = f"restaurant_dw.{nombre}"
    spark.sql(f"DROP TABLE IF EXISTS {tabla}")
    ruta = f"{WAREHOUSE}/{nombre}"
    if os.path.exists(ruta):
        shutil.rmtree(ruta, ignore_errors=True)
    df.write.mode("overwrite").saveAsTable(tabla)
    print(f"✅ {tabla}: {df.count()} filas guardadas.")

save_table(df_mom, "resultado_tendencias_mes_categoria")
save_table(df_top, "resultado_top_platos_mes")

spark.stop()